In [2]:
import json
from typing import List, Dict

import requests
from transformers import AutoTokenizer
from openai import OpenAI

c:\Users\germa\AppData\Local\miniconda3\envs\work_env\lib\site-packages\transformers\utils\generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


# Предобработка входных данных

В данном задании мы будем ходить в онлайн модель. Предлагается все также ходить в together.ai, т.к. они дают $5 кредита при регистрации.

Вначале давайте руками поиграемся с API, посмотрим, как походы в API соотносятся с тем, что мы делали в домашнем задании "Доступные LLM"

In [31]:
tokenizer = AutoTokenizer.from_pretrained("NousResearch/Meta-Llama-3.1-70B")

c:\Users\germa\AppData\Local\miniconda3\envs\work_env\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


## Ручное форматирование промпта - 5 баллов

Давайте попробуем собрать вход для llama3.1 руками, для этого допишем функцию `format_messages_to_prompt`.
Она принимает messages - массив словарей, где указаны роли и текст сообщений, а возвращает она текст в формате, который нужно подать модели.

Например для истории сообщений

```python
messages = [
    {"role": "system", "content": "Some system message"},
    {"role": "user", "content": "This is a message from the user"},
    {"role": "assistant", "content": "this is a mesage from the assistant"}
]
```

должен выдаваться итоговый промпт

```text
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Some system message<|eot_id|><|start_header_id|>user<|end_header_id|>

This is a message from the user<|eot_id|><|start_header_id|>assistant<|end_header_id|>

this is a mesage from the assistant<|eot_id|>
```

Что важно:
1. Текст начинается со спецтокена bos
2. Дальше идет заголовок start_header_id + end_header_id, которые содержат роль
3. Дальше после \n\n идет текст, заканчивающийся на eot_id
4. Дальше следующий заголовок с новой ролью и т.д.

**Важно** - в данной функции нельзя использовать `tokenizer.apply_chat_template`

In [32]:
def format_messages_to_prompt(messages: List[Dict[str, str]]) -> str:
    out = "<|begin_of_text|>"
    for message in messages:
        role = message["role"]
        content = message["content"]
        out += f"<|start_header_id|>{role}<|end_header_id|>\n\n"
        out += f"{content}<|eot_id|>"
    return out


messages = [
    {"role": "system", "content": "Some system message"},
    {"role": "user", "content": "This is a message from the user"},
    {"role": "assistant", "content": "this is a mesage from the assistant"}
]



reference_text = """<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Some system message<|eot_id|><|start_header_id|>user<|end_header_id|>

This is a message from the user<|eot_id|><|start_header_id|>assistant<|end_header_id|>

this is a mesage from the assistant<|eot_id|>"""


assert format_messages_to_prompt(messages) == reference_text

Мы также помним, что раньше у нас была `tokenizer.apply_chat_template`. Т.к. у нас неофициальный форк llama3.1, то chat_template в токенайзер нам не завезли, поэтому придется добавить его руками

In [33]:
chat_template = """
{{- bos_token }}
{%- if custom_tools is defined %}
    {%- set tools = custom_tools %}
{%- endif %}
{%- if not tools_in_user_message is defined %}
    {%- set tools_in_user_message = true %}
{%- endif %}
{%- if not date_string is defined %}
    {%- set date_string = "26 Jul 2024" %}
{%- endif %}
{%- if not tools is defined %}
    {%- set tools = none %}
{%- endif %}

{#- This block extracts the system message, so we can slot it into the right place. #}
{%- if messages[0]['role'] == 'system' %}
    {%- set system_message = messages[0]['content']|trim %}
    {%- set messages = messages[1:] %}
{%- else %}
    {%- set system_message = "" %}
{%- endif %}

{#- System message + builtin tools #}
{{- "<|start_header_id|>system<|end_header_id|>\n\n" }}
{%- if builtin_tools is defined or tools is not none %}
    {{- "Environment: ipython\n" }}
{%- endif %}
{%- if builtin_tools is defined %}
    {{- "Tools: " + builtin_tools | reject('equalto', 'code_interpreter') | join(", ") + "\n\n"}}
{%- endif %}
{{- "Cutting Knowledge Date: December 2023\n" }}
{{- "Today Date: " + date_string + "\n\n" }}
{%- if tools is not none and not tools_in_user_message %}
    {{- "You have access to the following functions. To call a function, please respond with JSON for a function call." }}
    {{- 'Respond in the format {"name": function name, "parameters": dictionary of argument name and its value}.' }}
    {{- "Do not use variables.\n\n" }}
    {%- for t in tools %}
        {{- t | tojson(indent=4) }}
        {{- "\n\n" }}
    {%- endfor %}
{%- endif %}
{{- system_message }}
{{- "<|eot_id|>" }}

{#- Custom tools are passed in a user message with some extra guidance #}
{%- if tools_in_user_message and not tools is none %}
    {#- Extract the first user message so we can plug it in here #}
    {%- if messages | length != 0 %}
        {%- set first_user_message = messages[0]['content']|trim %}
        {%- set messages = messages[1:] %}
    {%- else %}
        {{- raise_exception("Cannot put tools in the first user message when there's no first user message!") }}
{%- endif %}
    {{- '<|start_header_id|>user<|end_header_id|>\n\n' -}}
    {{- "Given the following functions, please respond with a JSON for a function call " }}
    {{- "with its proper arguments that best answers the given prompt.\n\n" }}
    {{- 'Respond in the format {"name": function name, "parameters": dictionary of argument name and its value}.' }}
    {{- "Do not use variables.\n\n" }}
    {%- for t in tools %}
        {{- t | tojson(indent=4) }}
        {{- "\n\n" }}
    {%- endfor %}
    {{- first_user_message + "<|eot_id|>"}}
{%- endif %}

{%- for message in messages %}
    {%- if not (message.role == 'ipython' or message.role == 'tool' or 'tool_calls' in message) %}
        {{- '<|start_header_id|>' + message['role'] + '<|end_header_id|>\n\n'+ message['content'] | trim + '<|eot_id|>' }}
    {%- elif 'tool_calls' in message %}
        {%- if not message.tool_calls|length == 1 %}
            {{- raise_exception("This model only supports single tool-calls at once!") }}
        {%- endif %}
        {%- set tool_call = message.tool_calls[0].function %}
        {%- if builtin_tools is defined and tool_call.name in builtin_tools %}
            {{- '<|start_header_id|>assistant<|end_header_id|>\n\n' -}}
            {{- "<|python_tag|>" + tool_call.name + ".call(" }}
            {%- for arg_name, arg_val in tool_call.arguments | items %}
                {{- arg_name + '="' + arg_val + '"' }}
                {%- if not loop.last %}
                    {{- ", " }}
                {%- endif %}
                {%- endfor %}
            {{- ")" }}
        {%- else  %}
            {{- '<|start_header_id|>assistant<|end_header_id|>\n\n' -}}
            {{- '{"name": "' + tool_call.name + '", ' }}
            {{- '"parameters": ' }}
            {{- tool_call.arguments | tojson }}
            {{- "}" }}
        {%- endif %}
        {%- if builtin_tools is defined %}
            {#- This means we're in ipython mode #}
            {{- "<|eom_id|>" }}
        {%- else %}
            {{- "<|eot_id|>" }}
        {%- endif %}
    {%- elif message.role == "tool" or message.role == "ipython" %}
        {{- "<|start_header_id|>ipython<|end_header_id|>\n\n" }}
        {%- if message.content is mapping or message.content is iterable %}
            {{- message.content | tojson }}
        {%- else %}
            {{- message.content }}
        {%- endif %}
        {{- "<|eot_id|>" }}
    {%- endif %}
{%- endfor %}
{%- if add_generation_prompt %}
    {{- '<|start_header_id|>assistant<|end_header_id|>\n\n' }}
{%- endif %}
""".strip()
tokenizer.chat_template = chat_template

## Автоматическая сборка промпта - 5 баллов

Давайте вспомним теперь на деле, как используется chat_template! Попробуем использовать функцию `tokenizer.apply_chat_template`

In [34]:
messages = [
    {"role": "system", "content": "Some system message"},
    {"role": "user", "content": "This is a message from the user"},
    {"role": "assistant", "content": "this is a mesage from the assistant"}
]

prompt = tokenizer.apply_chat_template(messages, tokenize=False)# Ваш код здесь

In [35]:
reference_prompt = """<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

Some system message<|eot_id|><|start_header_id|>user<|end_header_id|>

This is a message from the user<|eot_id|><|start_header_id|>assistant<|end_header_id|>

this is a mesage from the assistant<|eot_id|>"""

assert prompt == reference_prompt

Обратите внимание, что в заданном chat_template указаны Cutting Knowledge Date, т.е. до данные до какого периода видела модели, и Today Date - захардкоженная дата текущего диалога.

**Вопрос, обязательно напишите свой ответ здесь!**
На что влияет аргумент `add_generation_prompt` в функции `tokenizer.apply_chat_template`? Зачем его использовать?

### Определяет, добавлять ли в самый конец строки маркер того, что дальше должна отвечать моделька или нет
 - True для генерации
 - False для тренировки, когда от ассистента уже есть готовый ответ

## Походы в API - 10 баллов

Теперь давайте посмотрим, как можно ходить в API. Для примера мы будем ходить в together.ai, который щедро предоставляет $5 всем зарегистрировавшимся. Вообще говоря различных провайдеров много, API у них у всех очень похожий, т.к. все мимикрируют под OpenAI.

### Ключ временный делал если что, поэтмоу и его запушил(если все же забыл удалить)

In [ ]:
API_KEY = ...
MODEL = "poolside/laguna-m.1:free"

Есть несколько способов сходить в API. Можно ходить напрямую через библиотеку **requests**. Допишите post запрос в `url` с данными `data` и заголовками `headers`.

In [37]:
def send_request(prompt: str, model: str = "poolside/laguna-m.1:free"):
  response = requests.post(
    url="https://openrouter.ai/api/v1/chat/completions",
    headers={
      "Authorization": f"Bearer {API_KEY}",
    },
    json={
      "model": model,
      "messages": [
          {
              "role": "user",
              "content": prompt
          }
      ]
    }
  )
  
  if response.status_code == 200:
    data = response.json()  
    
    reply = data["choices"][0]["message"]["content"]
    
    print("Ответ нейросети:")
    print(reply)
    return reply
  else:
    print(f"Ошибка {response.status_code}: {response.text}")
    return None

Мы подали messages, дальше они каким-то образом собрались в promt и подались модели. Мы не знаем, какой промпт используется на стороне провайдера. Вспомним про Today Date из предыдущего пункта задания - использует ли его together? Обновляют ли они его сегодняшним днем или оставляют 26 июля? Если обновляют, то по какому часовому поясу?

Чтобы ответы на эти и многие другие вопросы не мучали нас по ночам, можно использовать prompt формат, а именно подать модели текст напрямую на генерацию. Давайте для этого используем `tokenizer.apply_chat_template`. Модель будет принимать текст ровно так, как вы его подадите, без каких-либо предобработок. Подумайте, нужно ли вам использовать аргумент `add_generation_prompt`?

Чтобы послать запрос напрямую, нужно в предыдущем запросе убрать messages, который представляет из себя список словарей, и послать поле prompt - строку с промптом для модели.

In [38]:
headers = {
    'Authorization': 'Bearer' + API_KEY,
    'Content-Type': 'application/json' 
}
url = "https://openrouter.ai/api/v1/completions"

messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "What is the capital of Britain?"}
]

prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

reply = send_request(prompt)

assert "london" in reply.lower() and "assistant" not in reply.lower()

Ответ нейросети:

The capital of the United Kingdom (often referred to as "Britain" in common usage) is **London**. 

If you meant a specific part of the UK:
- **England's** capital is London.
- **Scotland's** capital is Edinburgh.
- **Wales's** capital is Cardiff.
- **Northern Ireland's** capital is Belfast.

Let me know if you need further clarification!



## Клиент - 5 баллов

Теперь мы понимаем общую схему взаимодействия с провайдером - они предоставляют апи, куда можно посылать или промтп или историю диалога. При посылке промпта вся ответственность за формат ложится на нас, при посылке messages форматтинг происходит на стороне провайдера, но мы не всегда представляем, как он работает. Выбор в пользу того или иного варианта всегда остается на вас.

Мы использовали выше библиотеку requests, чтобы послать HTTP-запрос на сервера together, однако есть способ и проще - python client. Давайте познакомимся с ним поближе. Для этого давайте используем функцию `client.chat.completions.create`. Также давайте добавим опции сэмплинга, которые в этой функции поддержаны. Их можно посылать и в запросах через requests, но мы здесь и далее будем пользоваться клиентом.
* top_k = 100
* temperature = 0.5
* top_p = 0.9
* repetition_penalty = 1.05

In [10]:
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=API_KEY
)

In [ ]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "What is the capital of Britain?"}
]

response = client.chat.completions.create(
    model=MODEL,
    messages=messages,
    # top_k=100,
    temperature=0.5,
    top_p=0.9,
    frequency_penalty=1.05
)

response_text = response.choices[0].message.content
print(response_text)
assert "london" in response_text.lower()


The capital of Britain is **London**. 

Note: While "Britain" technically refers to the island comprising England, Scotland, and Wales, it is often used colloquially to mean the United Kingdom (UK), whose capital is also London. If you were referring specifically to the UK, the answer remains the same.



Аналогично посылать просто prompt можно через `client.completions.create`.

## Tools - 5 баллов

Давайте теперь посмотрим, как можно использовать tools в связке с моделями. У нас есть функция, которая входит в базу данных и получает информацию о юзере. Базы данных, конечно же, у нас никакой нет, но у нас есть некоторая функция, которая эмулирует это поведение, так что давайте попробуем ее описать.


In [12]:
def get_user_info_from_db(person_name: str) -> Dict[str, str]:
    database = {
        "ilya": {
            "job": "Software Developer",
            "pets": "dog",
        },
        "farruh": {
            "job": "Senior Data & Solution Architect",
            "hobby": "travelling, hiking",
        },
        "timur": {
            "job": "DeepSchool Founder",
            "city": "Novosibirsk",
        }
    }
    no_info = {"err": f"No info about {person_name}"}
    return database.get(person_name.lower(), no_info)

print(get_user_info_from_db("Timur"))
print(get_user_info_from_db("Aboba"))

{'job': 'DeepSchool Founder', 'city': 'Novosibirsk'}
{'err': 'No info about Aboba'}


Давайте попробуем описать эту функцию в формате json, чтобы модель могла ее увидеть!
Заполните поля в определении дальше

In [13]:
get_user_info_from_db_tool = {
    "type": "function",
    "function": {
        "name": "get_user_info_from_db",
        "description": "Получает информацию о человеке из базы данных: должность, город проживания.\
            Возвращает словарь с данными или сообщение об ошибке, если человека нет в базе.",
        "parameters":{
            "type": "object",
            "properties": {
                "person_name": {
                    "type": "string",
                    "description": "Имя человека, по которому искать информацию в базе данных (например, 'Timur', 'Ilya', 'Farruh')"
                }
            },
            "required": ["person_name"]
        }
    }
}

Теперь давайте подадим это описание в `tokenizer.apply_chat_template`. Обратите внимание на его аргумент `tools`! Не забудьте `add_generation_prompt`, если он нужен.

In [14]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "What do you know about Ilya?"}
]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, tools=[get_user_info_from_db_tool])# Ваш код здесь
print(prompt)

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

You are a helpful assistant<|eot_id|><|start_header_id|>user<|end_header_id|>

What do you know about Ilya?<|eot_id|><|start_header_id|>assistant<|end_header_id|>




Давайте пошлем наш запрос в модель. На выбор 2 модели, если не будет работать с 8b, то предлагается посылать в 70b.
Для данного запроса для 8b был подобран работающий `seed=9706540181089681000`, который можно подать в функцию.

Давайте воспользуемся `client.completions.create` для генерации ответа от модели.

In [15]:
model_8b = "meta-llama/llama-3.1-8b-instruct:free"
model_70b = "meta-llama/llama-3.3-70b-instruct:free"

In [16]:
# response_8b = client.completions.create ... # Ваш код здесь
# response_70b = client.completions.create... # Ваш код здесь

# print(response_8b.choices[0].text)
# print(response_70b.choices[0].text)

response = client.chat.completions.create(
    model="openai/gpt-oss-120b:free",
    messages=[
        {"role": "system", "content": "You are a helpful assistant"},
        {"role": "user", "content": "What do you know about Ilya?"}
    ],
    tools=[get_user_info_from_db_tool],
    temperature=0.5,
    top_p=0.9,
)

message = response.choices[0].message

print(f"Content: {message.content}")
print(f"Tool calls: {message.tool_calls}")

if message.tool_calls:
    for tool_call in message.tool_calls:
        print(f"\nFunction name: {tool_call.function.name}")
        print(f"Arguments: {tool_call.function.arguments}")

Content: None
Tool calls: [ChatCompletionMessageFunctionToolCall(id='chatcmpl-tool-9d558de08ff613eb', function=Function(arguments='{\n  "person_name": "Ilya"\n}', name='get_user_info_from_db'), type='function', index=0)]

Function name: get_user_info_from_db
Arguments: {
  "person_name": "Ilya"
}


Если все хорошо, то мы получили ответ от модели, который выглядит как некоторый структурированный вывод, который можно использовать для вызова модели. Давайте попробуем написать функцию, которая принимает ответ модели в "сыром виде", выбирает, какую функцию с какими аргументами вызвать.

Здесь нам поможет FUNCTION_REGISTRY и то, что параметры в функцию можно передавать как словарь, например так
```python
def foo(a, b, c):
    print(a, b, c)

obj = {'b':10, 'c':'lee'}

foo(100, **obj)
```

In [17]:
FUNCTION_REGISTRY = {"get_user_info_from_db": get_user_info_from_db}
# На случай, если модель не генерит function call
reference_answer = """{"name": "get_user_info_from_db", "parameters": {"person_name": "Ilya"}}"""


def parse_function_call(model_answer):
    # Ваш код здесь
    # 1. Проверим, является ли это function call.
    # 2. Вызов нужной функции с указанными аргументами
    
    message = model_answer.choices[0].message
    
    print(f"Content: {message.content}")
    print(f"Tool calls: {message.tool_calls}")

    if message.tool_calls:
        for tool_call in message.tool_calls:
            called_tool = tool_call.function.name
            called_tool_args_str = tool_call.function.arguments
            
            print(f"\nFunction name: {called_tool}")
            print(f"Arguments: {called_tool_args_str}")

            called_tool_args = json.loads(called_tool_args_str)
            print(f"Parsed arguments: {called_tool_args}")

            function = FUNCTION_REGISTRY[called_tool]
            result = function(**called_tool_args)
            print(f"Result: {result}")
            
            return result
        
assert parse_function_call(response) == get_user_info_from_db("Ilya")

Content: None
Tool calls: [ChatCompletionMessageFunctionToolCall(id='chatcmpl-tool-9d558de08ff613eb', function=Function(arguments='{\n  "person_name": "Ilya"\n}', name='get_user_info_from_db'), type='function', index=0)]

Function name: get_user_info_from_db
Arguments: {
  "person_name": "Ilya"
}
Parsed arguments: {'person_name': 'Ilya'}
Result: {'job': 'Software Developer', 'pets': 'dog'}


Теперь давайте попробуем объединить все это в историю диалога и сгенерировать моделью финальный ответ.
Для этого в messages, где хранится наша история диалога нужно добавить
1. Вызов function call моделью с ролью ХХХ (это часть задания, напишите сами)
2. Ответ function call с ролью tool

После этого данный промпт нужно послать модели снова, чтобы получить финальный ответ.
Для этого опять используем `tokenizer.apply_chat_template` и `client.completions.create`.

В зависимости от модели может понадобиться убрать tools (на 8b, 70b должна справиться). Для 8b опять же подобран seed=2017684582943914000

In [19]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "What do you know about Ilya?"}
]
messages.append({
    "role": "assistant",
    "content": message.content,  # None в нашем случае
    "tool_calls": [
        {
            "id": tool_call.id,
            "type": "function",
            "function": {
                "name": tool_call.function.name,
                "arguments": tool_call.function.arguments
            }
        }
    ]
})

function_result = parse_function_call(response)

# Добавляем ответ tool
messages.append({
    "role": "tool",
    "tool_call_id": tool_call.id,
    "content": json.dumps(function_result, ensure_ascii=False)
})

prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

response_8b = client.completions.create(
    model=MODEL,  # или model_8b
    prompt=prompt,
    temperature=0.5,
    top_p=0.9,
    max_tokens=500,
    seed=2017684582943914000  # seed из задания для 8b модели
)

print(response_8b.choices[0].text)

Content: None
Tool calls: [ChatCompletionMessageFunctionToolCall(id='chatcmpl-tool-9d558de08ff613eb', function=Function(arguments='{\n  "person_name": "Ilya"\n}', name='get_user_info_from_db'), type='function', index=0)]

Function name: get_user_info_from_db
Arguments: {
  "person_name": "Ilya"
}
Parsed arguments: {'person_name': 'Ilya'}
Result: {'job': 'Software Developer', 'pets': 'dog'}

I found some information about Ilya! According to the database, Ilya is a **Software Developer** and has a **dog** as a pet. 

Would you like to know more details about Ilya, or is there something specific you're curious about?



Теперь давайте посмотрим на chat-API, как обрабатываются function calls там?
Используем для этого уже знакомый `client.chat.completions.create`, обратим внимание на аргумент tools внутри него. Здесь рекомендуется использовать 70b модель. На всякий случай работающий seed=14157400267283583000

In [ ]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "What do you know about Ilya?"}
]
response = client.chat.completions.create(
    messages=messages,
    model=MODEL,
    frequency_penalty=0.5,
    tools=[get_user_info_from_db_tool],
    top_p=0.9) # Ваш код здесь

print(response)
print(response.choices[0].message.tool_calls)

ChatCompletion(id='gen-1781025374-mwYLX2cZ2LzBIpa0u7gq', choices=[Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content="\nI'll find out information about Ilya.\n\n", refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='chatcmpl-tool-0c3cf1538bf244bba0b773b31af121e7', function=Function(arguments='{"person_name": "Ilya"}', name='get_user_info_from_db'), type='function', index=0)], reasoning='\nThe user asks: "What do you know about Ilya?" I should check if there is a tool to get information about a person from a database. There is get_user_info_from_db. I can use that to find info about Ilya.\n', reasoning_details=[{'type': 'reasoning.text', 'text': '\nThe user asks: "What do you know about Ilya?" I should check if there is a tool to get information about a person from a database. There is get_user_info_from_db. I can use that to find info about Ilya.\n', 'format

Мы можем видеть, что у нас не работает предыдущий подход с полем `content`, однако должно было появиться поле `tool_calls`, которое содержит в себе информацию о вызове инструмента

In [22]:
response.choices[0].message.tool_calls

[ChatCompletionMessageFunctionToolCall(id='chatcmpl-tool-0c3cf1538bf244bba0b773b31af121e7', function=Function(arguments='{"person_name": "Ilya"}', name='get_user_info_from_db'), type='function', index=0)]

# Использование библиотек

Теперь, когда мы руками прошли весь пути обработки function call можно посмотреть уже на готовые инструменты.
Мы много чего сделали руками:
1. Писали описание функции
2. Обрабатывали ответ
3. Вызывали функцию
4. Возвращали все это в модель

Давайте теперь посмотрим, как оно работает в библиотеках!

**NB** - библиотеки развиваются и вполне, возможно, что к концу курса те интерфейсы, которые мы используем в этом домашнем задании будут уже неактуальны, но я уверен, что знаний и принципов, полученных из этих заданий хватит, чтобы адаптироваться к будущим вызовам!

# LangChain - 5 баллов

In [1]:
import os
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool

from pprint import pprint

Давайте ознакомимся с langchain-интеграцией together.ai

In [26]:
os.environ["OPEN_ROUTER_API_KEY"] = API_KEY


llm = ChatOpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=API_KEY,
    model=MODEL,
    temperature=0.5
)

In [27]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "What do you know about Ilya?"}
]
response = llm.invoke(messages)
print(response.content)


I need to clarify which Ilya you're referring to. There are several notable people named Ilya, including:

1. Ilya Prigogine - Nobel laureate in Chemistry known for his work on dissipative structures and irreversibility
2. Ilya Repin - Russian realist painter
3. Ilya Sutskever - Computer scientist, co-founder of OpenAI
4. Ilya Prigogine - Belgian physical chemist (Nobel Prize winner)
5. Ilya Muromets - Legendary figure in Russian folklore

Could you please specify which Ilya you're asking about?



Теперь, когда мы разобрались, как базово работать с langchain, давайте попробуем добавить инструментов. Чтобы нам было не так скучно, давайте напишем новую функцию, которая считает "волшебную операцию".

Эта функция принимает 2 строки, возвращает строку строку b в обратном порядке, сконкатенированную со строкой a. Допишите эту функцию.

In [ ]:
def magic_operation(a, b):
    
    return b[::-1] + a

assert magic_operation("456", "321") == "123456"

Теперь давайте обернем эту функцию в декоратор tool из langchain, аннотируем типы и допишем docstring. После этого можно будет автоматически сгенерировать описани функции в function call формате!

In [49]:
@tool
def magic_operation_tool(a: str, b: str) -> str: # аннотации типов
    """
        Выполняет «волшебную операцию» над двумя строками: разворачивает вторую строку (b) в обратном порядке и конкатенирует с первой строкой (a).
        
        Args:
            a: Первая строка, которая пойдёт в конец результата.
            b: Вторая строка, которая будет развёрнута и приписана к началу результата.
        
        Returns:
            Строку, где сначала идёт перевёрнутая b, а затем a.
        
        Example:
            magic_operation("456", "321") -> "123456"
    """
    
    return magic_operation(a, b)

print(magic_operation_tool.args_schema.schema())

{'title': 'magic_operation_toolSchema', 'description': 'Выполняет «волшебную операцию» над двумя строками: разворачивает вторую строку (b) в обратном порядке и конкатенирует с первой строкой (a).\n\nArgs:\n    a: Первая строка, которая пойдёт в конец результата.\n    b: Вторая строка, которая будет развёрнута и приписана к началу результата.\n\nReturns:\n    Строку, где сначала идёт перевёрнутая b, а затем a.\n\nExample:\n    magic_operation("456", "321") -> "123456"', 'type': 'object', 'properties': {'a': {'title': 'A', 'type': 'string'}, 'b': {'title': 'B', 'type': 'string'}}, 'required': ['a', 'b']}


Теперь давайте попробуем подать запрос в нашу LLM и обогатить ее нашим function_call. Для этого нужна функция `llm.bind_tools`.

In [50]:
llm_with_tools = llm.bind_tools(tools=[magic_operation_tool])# Ваш код здесь

Теперь давайте как и раньше:
1. Сгенерируем ответ на messages
2. Проверим в ответе resp.tool_calls, вызовем нужный инструмент
3. Расширим messages ответом модели и ответом инструмента, сгенерируем финальный ответ.

In [71]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "Can you help me? Do not reveal the workings of magic operation, but give me the result of it for strings `456` and `321`"}
]
resp = llm_with_tools.invoke(messages)

pprint(resp)
pprint(resp.tool_calls)

messages.append(
    {
        "role": "assistant",
        "content": resp.content
    }
)

messages.append(
    {
        "role": "tool",
        "tool_call_id": resp.tool_calls[0]["id"],
        "content": str(magic_operation(**resp.tool_calls[0]["args"])) 
    }
)

AIMessage(content='\n\n', additional_kwargs={'tool_calls': [{'id': 'chatcmpl-tool-8b785750177348dfb4108147d7710bc1', 'function': {'arguments': '{"a": "456", "b": "321"}', 'name': 'magic_operation_tool'}, 'type': 'function', 'index': 0}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 95, 'prompt_tokens': 423, 'total_tokens': 518, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 44, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 9, 'cache_write_tokens': 0, 'video_tokens': 0}, 'cost': 0, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 0, 'upstream_inference_prompt_cost': 0, 'upstream_inference_completions_cost': 0}}, 'model_name': 'poolside/laguna-m.1-20260312:free', 'system_fingerprint': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='run-5d1e5359-4d6b-43c1-b8aa-66b75d6ae6a5-0', tool_calls=[{'name': 'magic_ope

In [75]:
assert len(messages) == 4

pprint(messages)

[{'content': 'You are a helpful assistant', 'role': 'system'},
 {'content': 'Can you help me? Do not reveal the workings of magic operation, '
             'but give me the result of it for strings `456` and `321`',
  'role': 'user'},
 {'content': '\n\n', 'role': 'assistant'},
 {'content': '123456',
  'role': 'tool',
  'tool_call_id': 'chatcmpl-tool-8b785750177348dfb4108147d7710bc1'}]


In [73]:
res = llm.invoke(messages).content
assert "123456" in res

In [76]:
pprint(res)

('\n'
 'The result of the magic operation for the strings `456` and `321` is:\n'
 '\n'
 '**123456**\n')


# LlamaIndex - 5 баллов

Аналогичный инструмент LlamaIndex. В ней не так хороша поддержка function calls не для OpenAI, поэтому придется забежать вперед и использовать ReActAgent.

In [4]:
from llama_index.llms.openai_like import OpenAILike
from llama_index.core.tools import FunctionTool
from llama_index.core.agent.workflow import ReActAgent
from llama_index.core.agent.react.formatter import ReActChatFormatter

In [17]:
llm = OpenAILike(
    model=MODEL, 
    api_key=os.getenv("OPEN_ROUTER_API_KEY", API_KEY),
    api_base="https://openrouter.ai/api/v1",
    is_chat_model=True,
    is_function_calling_model=False,
    context_window=4096,
    num_output=1024
)

Скопируйте magic_operation_tool из части с langchain сюда,  но без декоратора.

In [18]:
def magic_operation_tool(a: str, b: str) -> str: # аннотации типов
    """
        Выполняет «волшебную операцию» над двумя строками: разворачивает вторую строку (b) в обратном порядке и конкатенирует с первой строкой (a).
        
        Args:
            a: Первая строка, которая пойдёт в конец результата.
            b: Вторая строка, которая будет развёрнута и приписана к началу результата.
        
        Returns:
            Строку, где сначала идёт перевёрнутая b, а затем a.
        
        Example:
            magic_operation("456", "321") -> "123456"
    """
    
    return magic_operation(a, b)

Мы можем аналогично создать инструмент с помощью `FunctionTool.from_defaults`

In [19]:
magic_operation_tool_llamaindex = FunctionTool.from_defaults(magic_operation_tool)
pprint(magic_operation_tool_llamaindex.metadata)

ToolMetadata(description='magic_operation_tool(a: str, b: str) -> str\n'
                         '\n'
                         '        Выполняет «волшебную операцию» над двумя '
                         'строками: разворачивает вторую строку (b) в обратном '
                         'порядке и конкатенирует с первой строкой (a).\n'
                         '        \n'
                         '        Args:\n'
                         '            a: Первая строка, которая пойдёт в конец '
                         'результата.\n'
                         '            b: Вторая строка, которая будет '
                         'развёрнута и приписана к началу результата.\n'
                         '        \n'
                         '        Returns:\n'
                         '            Строку, где сначала идёт перевёрнутая b, '
                         'а затем a.\n'
                         '        \n'
                         '        Example:\n'
                         ' 

Давайте создадим ReActAgent: ему нужно передать tools, llm, memory=None и verbose=True

In [20]:
CUSTOM_REACT_SYSTEM_HEADER = """You are a helpful assistant. Do not reveal the internal workings of tools.

You have access to the following tools:
{tool_desc}

To use a tool, you MUST use the following exact format:
Thought: Current thought process.
Action: tool_name
Action Input: {{"arg1": "value1", "arg2": "value2"}}

When you have the result from the tool and are ready to answer the user, you MUST use this format:
Thought: I have the result from the tool.
Answer: Your final response to the user here.

DO NOT try to call the tool again if you already see its output in the history!
"""

react_formatter = ReActChatFormatter(
    system_header=CUSTOM_REACT_SYSTEM_HEADER
)


agent = ReActAgent(
    tools=[magic_operation_tool_llamaindex],
    llm=llm,
    react_chat_formatter=react_formatter,
    streaming=False,
    verbose=True
)

In [21]:
text = "Can you help me? Do not reveal the workings of magic operation, but give me the result of it for strings `456` and `321`"
response = await agent.run(user_msg=text)
print(str(response))

[tick] add: AgentWorkflowStartEvent(user_msg='Can you help me? Do not reveal the workings of mag...', chat_history=None, memory=None, max_iterations=None, early_stopping_method=None)
[init_run:0] started from AgentWorkflowStartEvent
[init_run:0] complete with AgentInput
[tick] add: AgentInput(input=[ChatMessage(role=<MessageRole.USER: 'user'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text='Can you help me? Do not reveal the workings of magic operation, but g...
[setup_agent:0] started from AgentInput
[setup_agent:0] complete with AgentSetup
[tick] add: AgentSetup(input=[ChatMessage(role=<MessageRole.USER: 'user'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text='Can you help me? Do not reveal the workings of magic operation, but g...
[run_agent_step:0] started from AgentSetup
[run_agent_step:0] complete with AgentOutput
[tick] add: AgentOutput(response=ChatMessage(role=<MessageRole.ASSISTANT: 'assistant'>, additional_kwargs={}, blocks=[ThinkingBlock(bl

# Agents - 10

Настала пора сделать своего агента!
Попробуем сделать финансового аналитика. Требования следующие:
бот должен по запросу данных о какой-либо компании смотреть самые большие изменения цены ее акций за последний месяц, после чего бот должен объяснить, с какой новостью это связано.

Предлагается не строить сложную систему с классификаторами, а отдать всю сложную работу агенту. Давайте посмотрим, какие API нам доступны.

Первым делом получение котировок - для этого нам поможет библиотека yfinance. По названию компании и периоду отчетности можно посмотреть открывающие цены на момент открытия и закрытия биржи.

In [1]:
import yfinance as yf

stock = yf.Ticker("AAPL") # посмотрим котировки APPLE
df = stock.history(period="1mo")
df[["Open", "Close"]]

,Open,Close
Date,,
2026-05-11 00:00:00-04:00,291.980011,292.679993
2026-05-12 00:00:00-04:00,292.559998,294.799988
2026-05-13 00:00:00-04:00,293.500000,298.869995
2026-05-14 00:00:00-04:00,299.820007,298.209991
2026-05-15 00:00:00-04:00,297.899994,300.230011
2026-05-18 00:00:00-04:00,300.239990,297.839996
2026-05-19 00:00:00-04:00,296.970001,298.970001
2026-05-20 00:00:00-04:00,298.179993,302.250000
2026-05-21 00:00:00-04:00,301.059998,304.989990


Для поиска новостей нам поможет https://newsapi.org/
Можно легко получить свой ключ за короткую регистрацию, дается 1000 запросов в день, каждый запрос может включать в себя ключевое слово и промежуток дат. По бесплатному апи ключу дается ровно 1 месяц, что нам подходит.

In [ ]:
api_key = ... # ваш API ключ здесь!
api_template = "https://newsapi.org/v2/everything?q={keyword}&apiKey={api_key}&from={date_from}"

articles = requests.get(api_template.format(keyword="Apple", api_key=api_key, date_from=" 2026-05-10")).json()
for article in articles["articles"]:
    if article["title"] != "[Removed]":
        print(article["title"])
        print(article["description"])
        break

Cameras get an Apple Intelligence boost in Apple Home
Apple Intelligence is coming to cameras connected to Apple Home. At WWDC, Apple announced that with iOS27, the Home app will use Apple Intelligence to analyze footage and generate descriptions summarizing what the camera saw. You can also search footage with …


Очень много статей заблокированы и имеют название `[Removed]`, нужно их отфильтровать. В оставшихся статьях будем брать только title (заголовок) и description (описание или краткий пересказ).

Вам необходимо реализовать [ReAct Agent](https://react-lm.github.io/). Особенность этого агента заключается в том, что он вначале формирует мысль, а потом вызывает действие (function call) для достижения какой-либо цели.

Что нужно сделать:
1. Описать и реализовать function call для определения, в какой день была самая большая разница в цене акций в момент открытия и закрытия биржи. Функция получает один аргумент - название акций компании (например AAPL для Apple), а выдает словарь с 2мя полями: с датой максимальной разницы в ценах и самой разницей в ценах.
2. Описать и реализовать function call для получения 5 релевантных новостей о компании. В качестве аргумента принимаются название компании и дата. Ваша задача - сходить в newsapi, получить новости и вернуть 5 случайных новостей, которые произошли не позже чем день торгов. Если новостей меньше 5, то верните столько, сколько получится.
3. После этого агент должен вернуть ответ, в котором постарается аргументировать изменения в цене.


Реализовывать агента можно любым удобным способом, в том числе взять готовые имплементации.
1. [LlamaIndex](https://docs.llamaindex.ai/en/stable/examples/agent/react_agent/) - вдобавок можно посмотреть предыдущее задание, где он уже используется.
2. [Langchain/Langgraph](https://langchain-ai.github.io/langgraph/how-tos/create-react-agent/#code)
3. Написать полностью свою реализацию


Не забудьте, что очень важно описать задачу в промпте: нужно сказать, какие цели у агента и что он должен сделать. У функций должны быть говорящие описания, чтобы LLM без лишних проблем поняла, какие есть функции и когда их использовать. По всем вопросам можно обращаться в наш телеграм-чат в канал "Tools & Agents".


In [21]:
import os
import re
import json
import random
import requests
from datetime import datetime, timedelta
from typing import List, Dict, Any, Literal, Optional
from pydantic import BaseModel, Field

import yfinance as yf
from langchain_openai import ChatOpenAI
from langchain_core.messages import BaseMessage, SystemMessage, HumanMessage, AIMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from typing_extensions import TypedDict, Annotated

class NewsAnalyticStatement(BaseModel):
    
    messages: Annotated[list, add_messages]
    ticker: str
    company_name: str
    target_date: str
    final_answer: str
    steps: int = 0
    
class FinancialAnalystAgent:
    def __init__(self, news_api_key: str, open_router_api_key: str, model_name: str ="poolside/laguna-m.1:free", max_steps: int = 6):
        
        self.max_steps = max_steps
        self.news_api_key = news_api_key
        
        self.llm = ChatOpenAI(
            model=model_name,
            api_key=open_router_api_key,
            base_url="https://openrouter.ai/api/v1",
            temperature=0.0
        )      

        self.system_prompt = """
            You are an expert Financial Analyst Agent. Your task is to find the biggest stock price change for a company within the last month, fetch historical news around that specific date, and explain the variance.

            You MUST respond ONLY with a single valid JSON object. Do not include any conversational text, markdown formatting, or triple backticks (```) outside the JSON structure.

            Every response must exactly follow this JSON schema:
            {
                "thought": "Your analytical calculation, logic, or planning process.",
                "action": "tool_name_here" or null,
                "action_input": {"arg_name": "value"} or null,
                "final_answer": "Your comprehensive financial explanation here" or null
            }

            Available Tools:
            1. "get_max_stock_change": Expects {"ticker": "str"}. Returns {"date": "YYYY-MM-DD", "max_difference": float}.
            2. "get_company_news": Expects {"company_name": "str", "date_str": "YYYY-MM-DD"}. Returns a list of news articles.

            Rules:
            1. If you need to gather data, set "action" to the tool name, provide "action_input", and set "final_answer" to null.
            2. If you obtain a date from 'get_max_stock_change', immediately follow up by calling 'get_company_news' for that date.
            3. Once you have all required data and are ready to explain the variance, set "action" and "action_input" to null, and put your full analysis into "final_answer".
            4. Never hallucinate data. Execute tools to discover reality.
        """
        
        self.graph = None
        
    def _get_max_stock_change(self, ticker: str) -> Dict[str, Any]:
        try:
            stock = yf.Ticker(ticker)
            df = stock.history(period="1mo")
            df["Diff"] = abs(df["Close"] - df["Open"])
            max_idx = df.Diff.idxmax()
            max_row = df.loc[max_idx]
            
            return {
                "date": max_idx.strftime('%Y-%m-%d'),
                "max_difference": round(float(max_row['Diff']), 2)
            }
        except Exception as e:
            return {"error": str(e)}
        
    def _get_company_news(self, company_name: str, date_str: str) -> List[str]:
        
        try:
            api_template = "https://newsapi.org/v2/everything?q={keyword}&apiKey={api_key}&from={date_from}"

            articles = requests.get(api_template.format(keyword=company_name, api_key=self.news_api_key, date_from=date_str)).json()
            verified = []
            for article in articles["articles"]:
                if article["title"] != "[Removed]":
                    verified.append(f"'title':{article['title']} 'description':{article['description']} 'published_date': {article['publishedAt']}")
            
            if not verified:
                return [f"No articles discovered for {company_name} on/before {date_str}"]
            
            return verified
        except Exception as e:
            return [f"Error fetching news metadata: {str(e)}"]
        
    def _call_model(self, state: NewsAnalyticStatement) -> Dict[str, Any]:
        """Отправляет текущую историю промптов в LLM."""
        
        messages = list(state.messages)
        
        if not messages or not isinstance(messages[0], SystemMessage):
            messages.insert(0, SystemMessage(content=self.system_prompt))
        
        print(f"\n--- [Step {state.steps + 1}] LLM call ---")
        response = self.llm.invoke(messages)
        
        try:
            parsed = self._parse_react_response(response.content)
            print(f"Thought: {parsed.get('thought')}")
            if parsed.get('action'):
                print(f"Intention: '{parsed.get('action')}' with args{parsed.get('action_input')}")
            if parsed.get('final_answer'):
                print(f"Final answer is ready")
        except Exception:
            print(f"Model sent row json, not str: {response.content}")        

        return {
            "messages": [response],
            "steps": state.steps + 1
        }
    
    def _parse_react_response(self, text: str) -> Dict[str, Any]:
            # Удаляем markdown блоки кода если есть
            text = text.strip()
            
            # Убираем ```json и ``` в начале и конце
            if text.startswith("```json"):
                text = text[7:]  # Убираем ```json
            if text.startswith("```"):
                text = text[3:]  # Убираем ```
            if text.endswith("```"):
                text = text[:-3]  # Убираем ``` в конце
            
            text = text.strip()
            
            # Парсим JSON
            try:
                return json.loads(text)
            except json.JSONDecodeError as e:
                print(f"JSON parsing error: {e}")
                print(f"Raw text: {text[:200]}...")
                return {
                    "thought": "Error parsing response",
                    "action": None,
                    "action_input": None,
                    "final_answer": "Failed to parse model response"
                }
                
    def _execute_tools(self, state: NewsAnalyticStatement) -> dict[str, Any]:
        
        last_message = state.messages[-1]
        parsed = self._parse_react_response(last_message.content)
        result = None

        tool_name = parsed.get("action", "")
        print(f"Tool execution: {tool_name}...")

        if "get_max_stock_change" in parsed["action"]:
            ticker = parsed["action_input"].get("ticker")
            if not ticker:
                ticker = state.ticker
            result = self._get_max_stock_change(ticker)
            print(f"[Result yfinance]: {result}")
            
        elif "get_company_news" in parsed["action"]:
            company_name = parsed["action_input"].get("company_name")
            date_str = parsed["action_input"].get("date_str")
            result = self._get_company_news(company_name, date_str)
            print(f"[Result NewsAPI]: Articles found: {len(result)}")
            
        else:
            result = f"Function with name {parsed['action']} doesnt exist."
            print(f"Error: {result}")
            
        result_str = json.dumps(result)
            
        return {
            "messages": [HumanMessage(result_str)],
        }
    
    def _condition_edge(self, state: NewsAnalyticStatement) -> Literal["execute_tools", "finalize", "__end__"]:
        
        if state.steps >= self.max_steps:
            return "finalize"
        
        last_message = state.messages[-1]
        parsed = self._parse_react_response(last_message.content)
        
        if parsed["final_answer"]:
            return "finalize"
        elif parsed["action"]:
            return "execute_tools"
        else:
            return "finalize"  
          
    def _finalize(self, state: NewsAnalyticStatement) -> dict[str, Any]:
        
        last_message = state.messages[-1]
        parsed = self._parse_react_response(last_message.content)
        
        final_answer = parsed["final_answer"]
        
        if state.steps >= self.max_steps and not final_answer:
            return {"final_answer": "The agent reached the maximum number of steps and found nothing."}
        
        return {
            "final_answer": final_answer
        }
    
    def compile_graph(self):
        
        workflow = StateGraph(NewsAnalyticStatement)
        
        workflow.add_node("call_model", self._call_model)
        workflow.add_node("execute_tools", self._execute_tools)
        workflow.add_node("finalize", self._finalize)
        
        workflow.add_edge(START, "call_model")
        workflow.add_conditional_edges(
            "call_model",
            path=self._condition_edge,
            path_map={
                "execute_tools": "execute_tools",
                "finalize": "finalize"
            }
        )
        workflow.add_edge("execute_tools", "call_model")
        workflow.add_edge("finalize", END)
        
        self.graph = workflow.compile()
    
    def run(self, ticker: str, company_name: str) -> str:
        
        initial_state = NewsAnalyticStatement(
            messages=[HumanMessage(content=f"Analyze the stock price change and news for {company_name} ({ticker}) within the last month.")],
            ticker=ticker,
            company_name=company_name,
            target_date="",
            final_answer="",
        )

        if self.graph is None:
            self.compile_graph()
            
        final_state = self.graph.invoke(initial_state)
        
        return final_state["final_answer"]

In [ ]:
NEWS_API_KEY = ...
API_KEY =...

In [23]:
agent = FinancialAnalystAgent(NEWS_API_KEY, os.getenv("OPEN_ROUTER_API_KEY",API_KEY), MODEL)

In [24]:
answer = agent.run("AAPL", "Apple")


--- [Step 1] LLM call ---
Thought: I need to first identify the biggest stock price change for Apple (AAPL) within the last month. I'll use the get_max_stock_change tool to find this information, then fetch relevant news for that date.
Intention: 'get_max_stock_change' with args{'ticker': 'AAPL'}
Tool execution: get_max_stock_change...
[Result yfinance]: {'date': '2026-06-09', 'max_difference': 9.73}

--- [Step 2] LLM call ---
Thought: I have the date of the biggest stock price change (2026-06-09) with a max difference of 9.73. Now I need to fetch the company news for Apple around this date to understand the variance.
Intention: 'get_company_news' with args{'company_name': 'Apple', 'date_str': '2026-06-09'}
Tool execution: get_company_news...
[Result NewsAPI]: Articles found: 97

--- [Step 3] LLM call ---
Thought: I have gathered all the necessary data. The biggest stock price change for Apple (AAPL) occurred on June 9, 2026, with a maximum difference of 9.73. This date coincides with

In [25]:
answer

"Apple (AAPL) experienced its largest stock price change of 9.73 on June 9, 2026, which directly corresponds to the company's WWDC 2026 keynote event. The primary driver of this significant variance was Apple's major AI strategy announcement, headlined by the introduction of Siri AI - a 'profoundly more capable and personal assistant powered by Apple Intelligence' with personal context, world knowledge, and onscreen awareness. This marked Apple's most substantial AI push to date, addressing nearly two years of anticipation since their initial AI promises. However, the stock movement was likely amplified by the controversy surrounding Apple's decision to withhold Siri AI from European markets, with the company explicitly stating that EU users 'won't be getting Siri AI anytime soon, if ever' and attempting to shift blame to EU regulations. This created a complex market reaction - positive sentiment from the long-awaited AI capabilities finally being delivered, but tempered by concerns ov